# Clasificación de Géneros de Películas

### Parte 2: Ingeniería de Características

---

Pipeline de preprocesamiento NLP: texto crudo → limpieza → lematización → vectorización TF-IDF → matriz binaria de géneros.

In [2]:
# Librerías
import warnings
warnings.filterwarnings('ignore')
import re, ast
import nltk
import numpy as np
import pandas as pd
from nltk.corpus import stopwords
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split

nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)

True

In [3]:
# Cargar datos
dataTraining = pd.read_csv('data/dataTraining.csv', encoding='UTF-8', index_col=0)
dataTesting  = pd.read_csv('data/dataTesting.csv',  encoding='UTF-8', index_col=0)

print(f'Entrenamiento: {dataTraining.shape} | Test: {dataTesting.shape}')

Entrenamiento: (7895, 5) | Test: (3383, 3)


### 1. Parseo y limpieza de duplicados

In [4]:
# Convertir la columna genres de string a lista (ast.literal_eval es seguro, no usa eval)
dataTraining['genres'] = dataTraining['genres'].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else x
)

# Convertir a tuplas para permitir hashing y detección de duplicados
dataTraining['genres'] = dataTraining['genres'].apply(
    lambda x: tuple(x) if isinstance(x, list) else x
)
print(f'Duplicados encontrados: {dataTraining.duplicated().sum()}')
dataTraining = dataTraining.drop_duplicates()

# Restaurar a listas
dataTraining['genres'] = dataTraining['genres'].apply(
    lambda x: list(x) if isinstance(x, tuple) else x
)
print(f'Registros finales: {len(dataTraining)}')

Duplicados encontrados: 1
Registros finales: 7894


### 2. Limpieza de texto

In [5]:
def remove_tags(text):
    text = re.sub(r'<.*?>', '', text)    # eliminar etiquetas HTML
    text = re.sub(r'http\S+', '', text)  # eliminar URLs
    text = re.sub(r'[\W_]', ' ', text)  # eliminar caracteres no alfanuméricos
    return text.lower()                  # normalizar a minúsculas

# Aplicar limpieza a entrenamiento y test
dataTraining['plot'] = dataTraining['plot'].apply(remove_tags)
dataTesting['plot']  = dataTesting['plot'].apply(remove_tags)

# Verificar que no queden plots vacíos tras la limpieza
vacios = (dataTraining['plot'].str.strip() == '').sum()
print(f'Plots vacíos: {vacios}')

Plots vacíos: 0


### 3. Eliminación de stopwords

In [6]:
# Cargar stopwords del inglés (el corpus está en inglés)
stop_words = set(stopwords.words('english'))

def remove_stopwords(text):
    # Filtrar palabras vacías que no aportan señal predictiva
    return ' '.join(w for w in text.split() if w not in stop_words)

dataTraining['plot'] = dataTraining['plot'].apply(remove_stopwords)
dataTesting['plot']  = dataTesting['plot'].apply(remove_stopwords)

### 4. Lematización

In [7]:
w_tokenizer = nltk.tokenize.WhitespaceTokenizer()
lemmatizer  = nltk.stem.WordNetLemmatizer()

def lemmatize_text(text):
    # Reducir cada token a su forma base (running→run, movies→movie)
    return ' '.join(lemmatizer.lemmatize(w) for w in w_tokenizer.tokenize(text))

dataTraining['plot'] = dataTraining['plot'].apply(lemmatize_text)
dataTesting['plot']  = dataTesting['plot'].apply(lemmatize_text)

print('Lematización completa. Ejemplo:')
print(dataTraining['plot'].iloc[0][:200])

Lematización completa. Ejemplo:
story single father take eight year old son work railroad drawbridge bridge tender day boy meet woman boarding train drug abuser bridge father go engine room tell son stay edge nearby lake ship come b


### 5. Variable objetivo — MultiLabelBinarizer

In [8]:
# Convertir listas de géneros a matriz binaria (n_películas × 24)
mlb = MultiLabelBinarizer()
y   = mlb.fit_transform(dataTraining['genres'])

# Columnas de salida en formato de predicción
cols = [f'p_{g}' for g in mlb.classes_]

print(f'Shape de y: {y.shape} → ({len(dataTraining)} películas × {len(mlb.classes_)} géneros)')
print(f'Géneros: {list(mlb.classes_)}')

Shape de y: (7894, 24) → (7894 películas × 24 géneros)
Géneros: ['Action', 'Adventure', 'Animation', 'Biography', 'Comedy', 'Crime', 'Documentary', 'Drama', 'Family', 'Fantasy', 'Film-Noir', 'History', 'Horror', 'Music', 'Musical', 'Mystery', 'News', 'Romance', 'Sci-Fi', 'Short', 'Sport', 'Thriller', 'War', 'Western']


**Implicación del desbalanceo en la variable objetivo:** Los 24 géneros tienen frecuencias muy desiguales en la matriz binaria resultante. En la columna `Drama`, cerca del 60 % de las filas son 1. En `News`, menos del 1 %. Este desbalanceo no puede corregirse con más datos — es una característica real del corpus. La consecuencia directa sobre la métrica: el ROC-AUC macro promedia el rendimiento sobre los 24 géneros, dándole el mismo peso a `Drama` (fácil, muchos ejemplos) que a `News` (difícil, pocos ejemplos). Un modelo que funcione brillantemente en los géneros mayoritarios pero falle en los minoritarios puede tener un AUC aparentemente bueno mientras oculta problemas reales de cobertura en las clases raras.

### 6. División entrenamiento / validación

In [9]:
# Split 80/20 con semilla fija para reproducibilidad
X_train, X_val, y_train, y_val = train_test_split(
    dataTraining['plot'], y, test_size=0.2, random_state=42
)
print(f'X_train: {len(X_train)} | X_val: {len(X_val)}')

X_train: 6315 | X_val: 1579


### 7. Vectorización TF-IDF

In [10]:
# TF-IDF con trigramas: el mejor rango de n-gramas según búsqueda de hiperparámetros
# ngram_range=(1,3) captura frases como "serial killer", "outer space"
tfidf = TfidfVectorizer(
    stop_words='english',
    max_features=20000,
    ngram_range=(1, 3)
)

X_train_tfidf = tfidf.fit_transform(X_train)  # ajustar vocabulario solo en train
X_val_tfidf   = tfidf.transform(X_val)         # aplicar el mismo vocabulario en val

print(f'Vocabulario: {len(tfidf.vocabulary_):,} términos')
print(f'Matriz train: {X_train_tfidf.shape} (dispersa)')

Vocabulario: 20,000 términos
Matriz train: (6315, 20000) (dispersa)


**Dos decisiones críticas de esta vectorización:**

1. **`ngram_range=(1,3)`** captura frases compuestas que definen géneros de forma no ambigua: `"serial killer"` (Crime/Horror), `"outer space"` (Sci-Fi), `"civil war"` (History/War), `"romantic comedy"` (Romance/Comedy). Un modelo de unigramas leería esos tokens de forma independiente, perdiendo la señal semántica que emerge de su combinación. El coste es un vocabulario más grande; el beneficio es una representación más expresiva del contenido narrativo.

2. **`fit` solo sobre `X_train`, `transform` sobre `X_val`** es crítico para evitar *data leakage*. Si se usara el vocabulario completo para calcular los pesos TF-IDF, los IDF de los términos reflejarían información del conjunto de validación y contaminarían el modelo. Es uno de los errores más comunes y sutiles en proyectos NLP.

La matriz resultante tiene una densidad inferior al 1%: cada sinopsis activa una fracción mínima de las 20000 columnas. Esta dispersión extrema explica por qué la Regresión Logística supera a los árboles de decisión; los árboles necesitan miles de splits para aproximar una frontera lineal en alta dimensión, mientras que LR la resuelve directamente en un solo paso de optimización.